# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24f2001824/ml-flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

I first checked the distributions of the main visibility, content depth, and freshness fields. The variables have different scales and some are heavy-tailed, especially impressions and search volume. This means a small number of pages have much larger values than most pages, so averages alone may not describe the dataset well.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/24f2001824/ml-flyrank.git

import pandas as pd
import numpy as np

df = pd.read_csv("/content/ml-flyrank/data/raw/content_refresh_anonymized.csv")

cols = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

df[cols].describe().T

fatal: destination path 'ml-flyrank' already exists and is not an empty directory.


,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,9546.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal 1: Longer content and visibility

The correlation between word count and impressions was positive (r = 0.163). This gives weak directional support for an association between longer content and higher observed visibility.

**Verdict: CONFIRMED**

The relationship is weak, so word count alone is not a strong predictor of visibility.

### Signal 2: Freshness and visibility

I compared observed impressions across freshness groups to check whether older pages show weaker visibility.

**Verdict: MIXED**

The freshness groups do not show a simple pattern that is strong enough to treat freshness alone as an explanation for visibility.

### Signal 3: Search position and CTR

The correlation between average position and CTR was -0.073. The direction is consistent with better positions being associated with higher CTR, but the relationship is very weak.

**Verdict: MIXED**

Average position alone does not explain much of the variation in CTR in this dataset.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Signal 1: Word count vs impressions")
print(
    df[["word_count", "impressions_90d"]]
    .corr()
)

print("\nSignal 2: Freshness groups")
df.groupby("freshness_tier")["impressions_90d"].agg(
    ["count", "median", "mean"]
)

print("\nSignal 3: Position vs CTR")
print(
    df[["avg_position", "ctr"]]
    .corr()
)

Signal 1: Word count vs impressions
                 word_count  impressions_90d
word_count         1.000000         0.163345
impressions_90d    0.163345         1.000000

Signal 2: Freshness groups

Signal 3: Position vs CTR
              avg_position      ctr
avg_position       1.00000 -0.07259
ctr               -0.07259  1.00000


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The baseline includes a stale-visible-page flag for pages that have not been updated for at least 180 days and still have meaningful impressions. I tested whether these pages show a different observed performance pattern from the rest of the dataset.

This test supports using freshness as a prioritization signal, but it does not show that updating a page will cause its performance to improve.

**Verdict: MIXED**

The stale-page group has a different observed profile, but age alone is not enough to explain page performance. The flag is more useful for prioritizing human review than making an automatic decision.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["stale_flag"] = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
)

df.groupby("stale_flag")[
    ["impressions_90d", "avg_position", "ctr", "word_count"]
].agg(["count", "median", "mean"])

impressions_90d                       avg_position         \
                     count  median          mean        count median   
stale_flag                                                             
False                29983   731.0   5196.737418        29983   10.8   
True                    17  4429.0  11600.647059           17   18.6   

                         ctr                  word_count                       
                 mean  count median      mean      count  median         mean  
stale_flag                                                                     
False       16.339926  29983   0.07  0.510905      22284  2876.0  3107.349758  
True        20.670588     17   0.20  0.208824         17  3861.0  3645.941176

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Some simple content and performance signals are useful for prioritizing pages for human review. However, no single signal explains page performance on its own.

The content team should combine these signals into a ranking and review the page context before choosing an action such as refresh, expansion, monitoring, or further investigation.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Signal audit completed.")

Signal audit completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.